# 🧪 Lab 03: Windows, Offsets, and Freshness 🪟⏱️

Treasury says daily balances cannot be added across dates. Growth wants a rolling customer count. Data Platform asks who is paying for the refresh pipeline. This lab models a rolling window, explains semiadditive snapshots, and optionally creates an aggregated materialization so you can inspect a real query rewrite.

Both features have compute, workspace, and feature-specific requirements; materialization requires DBR 17.3+ plus serverless Lakeflow availability. Confirm the current [feature matrix](https://docs.databricks.com/aws/en/uc-semantics/metric-views/feature-availability) and workspace configuration first. The source is the TPC-H sample orders table, which has a finite date range; adapt the filter to dates actually present.

This bundle passes `:catalog` and `:schema` into the SQL cells; an interactive run exposes them as parameter widgets. If the schema does not exist, the schema-creation cell requires `CREATE SCHEMA` on the parent catalog.

### 🎯 Mission objectives

You’ll prove that a trailing window has boundaries, that a balance isn’t a sale wearing a tie, and that materialization is a freshness-and-cost contract rather than a performance superstition. The lab is intentionally explicit about what it can prove on the current workspace and what remains a feature-gated template.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema);

### Check the sample’s calendar before you blame the window

TPC-H is a finite sample, not an infinite event stream. Run this small control first and keep the view’s date filter inside the returned range; otherwise an empty result can masquerade as a broken window.

In [ ]:
%sql
SELECT MIN(o_orderdate) AS sample_min_date,
       MAX(o_orderdate) AS sample_max_date,
       COUNT(*) AS source_rows
FROM samples.tpch.orders;

## 1. Give Growth a rolling seven-day customer count

The `date` field is the window order. `trailing 7 day` excludes the anchor row by default; use explicit inclusivity when your supported runtime allows it and the business rule needs it. This first query leaves out `semiadditive` so the rolling example works on the broadest supported window feature set; the Treasury template below calls out the newer runtime requirement for that modifier. The order field must be deterministic; do not use `rand()` or `current_timestamp()` unless you want every meeting to produce a different answer.

In [ ]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || '.' || :schema || '.rolling_customers_metric_view')
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Rolling distinct customers by order date."
source: samples.tpch.orders
filter: o_orderdate >= DATE'1998-01-01'
fields:
  - name: date
    expr: o_orderdate
  - name: order_month
    expr: DATE_TRUNC('MONTH', o_orderdate)
measures:
  - name: rolling_7_day_customers
    expr: COUNT(DISTINCT o_custkey)
    window:
      - order: date
        range: trailing 7 day
  - name: total_revenue
    expr: SUM(o_totalprice)
$$;

In [ ]:
%sql
SELECT date,
       MEASURE(rolling_7_day_customers) AS rolling_7_day_customers,
       MEASURE(total_revenue) AS total_revenue
FROM IDENTIFIER(:catalog || '.' || :schema || '.rolling_customers_metric_view')
GROUP BY date
ORDER BY date;

## 2. Treasury objects: balances are semiadditive

A balance adds across accounts on one date, but you must not add Monday's balance to Tuesday's and call it a larger cash position. A basic pattern is `SUM(balance)` with a `current` window ordered by date and `semiadditive: last`; current Databricks feature notes list the `semiadditive` modifier for DBR 19 and above, so check the matrix before running this template. For period comparisons, define shifted base measures with a supported dated `offset` and compose a delta or rate. A month offset is usually more faithful than 365 days for year-over-year comparisons.

The commented YAML below is a template, not a runnable SQL statement. Confirm the exact runtime syntax and test anchor boundaries with known dates before putting it in the minutes.

In [ ]:
%sql
-- Example YAML fragment for a snapshot-balance measure:
-- measures:
--   - name: ending_balance
--     expr: SUM(balance)
--     window:
--       - order: snapshot_date
--         range: current
--         semiadditive: last

## 3. Optional: create a monthly aggregate

Data Platform has agreed to try one small optimization, provided the request has a query plan and a cost owner. The next cell creates a separate Metric View with a monthly revenue aggregate. Creation starts a managed Lakeflow pipeline and its initial refresh asynchronously, which incurs pipeline and storage costs. Run it only when serverless Lakeflow is available and you want to try materialization. The schedule is omitted for this short lab, so later refreshes must be triggered manually. The Metric View remains queryable from its source while the first refresh runs.

The aggregate includes only additive `total_revenue` at `order_month`. The rolling distinct-customer measure is non-additive and windowed, so it is left out.

In [ ]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || '.' || :schema || '.monthly_revenue_materialized_metric_view')
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
source: samples.tpch.orders
filter: o_orderdate >= DATE'1998-01-01'
fields:
  - name: order_month
    expr: DATE_TRUNC('MONTH', o_orderdate)
measures:
  - name: total_revenue
    expr: SUM(o_totalprice)
materialization:
  mode: relaxed
  materialized_views:
    - name: revenue_by_month
      type: aggregated
      dimensions:
        - order_month
      measures:
        - total_revenue
$$;

## 4. Check whether the optimization made the minutes

Wait for the initial managed pipeline refresh to finish. Then inspect the query plan and look for the Metric View materialization node. If no materialization is eligible or ready, the query can fall back to source computation. Relaxed rewrite uses the last refreshed snapshot without checking freshness.

```sql
EXPLAIN EXTENDED
SELECT order_month, MEASURE(total_revenue)
FROM IDENTIFIER(:catalog || '.' || :schema || '.monthly_revenue_materialized_metric_view')
GROUP BY order_month;
```

To refresh after the initial run, use the pipeline page linked from Catalog Explorer, or run `REFRESH MATERIALIZED VIEW IDENTIFIER(:catalog || '.' || :schema || '.monthly_revenue_materialized_metric_view')` on DBR 18.0+.

### Exercises
1. Compare an exclusive trailing seven-day window with an inclusive one, where available. Use a tiny dataset with known values to make the boundary obvious.
2. Draft a year-over-year measure with a `-12 month` offset and test month ends and leap years.
3. On DBR 19+, sketch a dense month index and a `range: all` alignment for year/month fields; explain what breaks when the index has gaps.
4. Identify a frequent dashboard query and propose the narrowest useful aggregate. Mark each measure as additive or exact-match-only.
5. Write down freshness, access-control, and refresh-cost requirements before enabling a production materialization.